# Etap 3: Serwer MiniMax-H3-Image (GGUF) na Kaggle (`GPU T4 x2` fallback, `Internet: ON`, bez PE)

Serwer MiniMax-H3-Image na Kaggle GPU T4 x2 (pula ~30h GPU/tyg.):
1. ComfyUI z `--lowvram` + DynamicVRAM (`comfy-aimdo`) + `Nif00/ComfyUI-GGUF` i domyslnym cache modeli (bez przeladowywania GGUF miedzy promptami; 1 lub 2 workery GPU 0 / GPU 1).
2. Brak Stage-1 Prompt Enhancera i brak cyklu `/free` — prompty od Codexa leca bezposrednio do `MiniMaxH3ImageConditioning` (6 krokow, `CFG 1.0`, obsluga `<Picture 1>`..`<Picture 16>`).
3. Pelne API bramy Bartosz OS (`:8000`, SSE, `/v1/idle/extend`, `/v1/shutdown`, tunel `cloudflared` + `BARTOSZ_OS_JSON`).

**Wskazowka:** Podepnij prywatny dataset `minimax-h3-image-t4-gguf` (z Etapu 2), aby przyspieszyc start.


In [ ]:
import base64, os, pathlib, sys

PKG_ROOT = pathlib.Path(os.environ.get('MINIMAX_PKG_ROOT', '/tmp/minimax_pkg'))
PKG_DIR = PKG_ROOT / 'minimax_kaggle'
PKG_DIR.mkdir(parents=True, exist_ok=True)

_MODULES_B64 = {
  "__init__.py": "IiIiTWluaU1heC1IMy1JbWFnZSBLYWdnbGUgVFBVIHY1ZS04IChJTlQ4LT5CRjE2IC8gQkYxNikgYW5kIFQ0IHgyIChHR1VGKSB0b29sa2l0LiIiIgoKZnJvbSBtaW5pbWF4X2thZ2dsZS5jb25zdGFudHMgaW1wb3J0ICgKICAgIENBTlZBU19NVUxUSVBMRSwKICAgIERFRkFVTFRfQ09NRllfRElSLAogICAgREVGQVVMVF9JTlBVVF9ESVIsCiAgICBERUZBVUxUX1ZFTlZfRElSLAogICAgREVGQVVMVF9XT1JLSU5HX0RJUiwKICAgIE1JTklNQVhfREVGQVVMVF9DRkcsCiAgICBNSU5JTUFYX0RFRkFVTFRfU0FNUExFUiwKICAgIE1JTklNQVhfREVGQVVMVF9TQ0hFRFVMRVIsCiAgICBNSU5JTUFYX0RFRkFVTFRfU1RFUFMsCiAgICBNT0RFTFMsCiAgICBQUkVTRVRTLAogICAgVDRfU0FGRV9DT01GWV9GTEFHUywKICAgIFRQVV9TQUZFX0NPTUZZX0ZMQUdTLAopCgpfX2FsbF9fID0gWwogICAgIkNBTlZBU19NVUxUSVBMRSIsCiAgICAiREVGQVVMVF9DT01GWV9ESVIiLAogICAgIkRFRkFVTFRfSU5QVVRfRElSIiwKICAgICJERUZBVUxUX1ZFTlZfRElSIiwKICAgICJERUZBVUxUX1dPUktJTkdfRElSIiwKICAgICJNSU5JTUFYX0RFRkFVTFRfQ0ZHIiwKICAgICJNSU5JTUFYX0RFRkFVTFRfU0FNUExFUiIsCiAgICAiTUlOSU1BWF9ERUZBVUxUX1NDSEVEVUxFUiIsCiAgICAiTUlOSU1BWF9ERUZBVUxUX1NURVBTIiwKICAgICJNT0RFTFMiLAogICAgIlBSRVNFVFMiLAogICAgIlQ0X1NBRkVfQ09NRllfRkxBR1MiLAogICAgIlRQVV9TQUZFX0NPTUZZX0ZMQUdTIiwKXQo=",
  "constants.py": "IiIiQXV0aG9yaXRhdGl2ZSBjb25zdGFudHMsIG1vZGVsIGNhdGFsb2csIGFuZCBLYWdnbGUgcnVudGltZSBkZWZhdWx0cyBmb3IgTWluaU1heC1IMy1JbWFnZS4KCk5vIGJ1aWx0LWluIFByb21wdCBFbmhhbmNlciAoUEUpIHdlaWdodHMgYXJlIGluY2x1ZGVkOiByaWNoIHByb21wdHMgYXJlIGF1dGhvcmVkIGV4dGVybmFsbHkKKGUuZy4gdmlhIENvZGV4KSBhbmQgcGFzc2VkIGRpcmVjdGx5IGludG8gYE1pbmlNYXhIM0ltYWdlQ29uZGl0aW9uaW5nYC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgTW9kZWxTcGVjOgogICAgIiIiTWV0YWRhdGEgZGVzY3JpYmluZyBhIHNpbmdsZSBNaW5pTWF4LUgzLUltYWdlIGFzc2V0IGFjcm9zcyBIdWdnaW5nIEZhY2UgcmVwb3MuIiIiCgogICAga2V5OiBzdHIKICAgIGhmX3JlcG9faWQ6IHN0cgogICAgaGZfcGF0aDogc3RyCiAgICBzdWJkaXI6IHN0cgogICAgZmlsZW5hbWU6IHN0cgogICAgYXBwcm94X2dpYjogZmxvYXQKICAgIGxvYWRlcl9raW5kOiBzdHIgICMgInNhZmV0ZW5zb3JzIiB8ICJnZ3VmIgogICAgZGVzY3JpcHRpb246IHN0cgoKCkhGX0NPTUZZX0lNQUdFX1JFUE8gPSAiQ29tZnktT3JnL01pbmlNYXgtSDMtSW1hZ2UiCkhGX0NPTUZZX0gzX1JFUE8gPSAiQ29tZnktT3JnL01pbmlNYXgtSDMiCkhGX0dHVUZfRElUX1JFUE8gPSAiQWJpcmF5L01pbmlNYXgtSDMtSW1hZ2UtR0dVRiIKSEZfR0dVRl9DTElQX1JFUE8gPSAibmlmMC9Rd2VuMy1WTC0zMkItSW5zdHJ1Y3QtTWluaU1heC1IMy1HR1VGIgoKTU9ERUxTOiBkaWN0W3N0ciwgTW9kZWxTcGVjXSA9IHsKICAgICMgLS0tIE5hdGl2ZSBDb21meS1Pcmcgc2FmZXRlbnNvcnMgKFRQVSB2NWUtOCAxMjggR2lCIEhCTSArIH4zMzAgR2lCIGhvc3QgUkFNKSAtLS0KICAgICJkaXRfaW50OCI6IE1vZGVsU3BlYygKICAgICAgICBrZXk9ImRpdF9pbnQ4IiwKICAgICAgICBoZl9yZXBvX2lkPUhGX0NPTUZZX0lNQUdFX1JFUE8sCiAgICAgICAgaGZfcGF0aD0iZGlmZnVzaW9uX21vZGVscy9taW5pbWF4X2gzX2ltYWdlX3BydW5lZF9pbnQ4X2NvbnZyb3Quc2FmZXRlbnNvcnMiLAogICAgICAgIHN1YmRpcj0iZGlmZnVzaW9uX21vZGVscyIsCiAgICAgICAgZmlsZW5hbWU9Im1pbmltYXhfaDNfaW1hZ2VfcHJ1bmVkX2ludDhfY29udnJvdC5zYWZldGVuc29ycyIsCiAgICAgICAgYXBwcm94X2dpYj0xOS41NCwKICAgICAgICBsb2FkZXJfa2luZD0ic2FmZXRlbnNvcnMiLAogICAgICAgIGRlc2NyaXB0aW9uPSJNaW5pTWF4LUgzLUltYWdlIFBydW5lZCAzMy4xQiBEaVQgKElOVDggQ29udlJvdCwgZGVxdWFudGl6ZWQgdG8gQkYxNiBvbiBUUFUgdjVlLTgpIiwKICAgICksCiAgICAiZGl0X2JmMTYiOiBNb2RlbFNwZWMoCiAgICAgICAga2V5PSJkaXRfYmYxNiIsCiAgICAgICAgaGZfcmVwb19pZD1IRl9DT01GWV9JTUFHRV9SRVBPLAogICAgICAgIGhmX3BhdGg9ImRpZmZ1c2lvbl9tb2RlbHMvbWluaW1heF9oM19pbWFnZV9wcnVuZWRfYmYxNi5zYWZldGVuc29ycyIsCiAgICAgICAgc3ViZGlyPSJkaWZmdXNpb25fbW9kZWxzIiwKICAgICAgICBmaWxlbmFtZT0ibWluaW1heF9oM19pbWFnZV9wcnVuZWRfYmYxNi5zYWZldGVuc29ycyIsCiAgICAgICAgYXBwcm94X2dpYj0zNy40NywKICAgICAgICBsb2FkZXJfa2luZD0ic2FmZXRlbnNvcnMiLAogICAgICAgIGRlc2NyaXB0aW9uPSJNaW5pTWF4LUgzLUltYWdlIFBydW5lZCAzMy4xQiBEaVQgKE5hdGl2ZSBCRjE2IGZvciBUUFUgdjVlLTgpIiwKICAgICksCiAgICAiY2xpcF9pbnQ4IjogTW9kZWxTcGVjKAogICAgICAgIGtleT0iY2xpcF9pbnQ4IiwKICAgICAgICBoZl9yZXBvX2lkPUhGX0NPTUZZX0lNQUdFX1JFUE8sCiAgICAgICAgaGZfcGF0aD0idGV4dF9lbmNvZGVycy9xd2VuM3ZsXzMyYl9taW5pbWF4X2gzX2ludDhfY29udnJvdC5zYWZldGVuc29ycyIsCiAgICAgICAgc3ViZGlyPSJ0ZXh0X2VuY29kZXJzIiwKICAgICAgICBmaWxlbmFtZT0icXdlbjN2bF8zMmJfbWluaW1heF9oM19pbnQ4X2NvbnZyb3Quc2FmZXRlbnNvcnMiLAogICAgICAgIGFwcHJveF9naWI9MjUuMjgsCiAgICAgICAgbG9hZGVyX2tpbmQ9InNhZmV0ZW5zb3JzIiwKICAgICAgICBkZXNjcmlwdGlvbj0iUXdlbjMtVkwtMzJCIChMMC00OSkgTWluaU1heC1IMyB0ZXh0L3Zpc2lvbiBlbmNvZGVyIChJTlQ4IENvbnZSb3QpIiwKICAgICksCiAgICAiY2xpcF9iZjE2IjogTW9kZWxTcGVjKAogICAgICAgIGtleT0iY2xpcF9iZjE2IiwKICAgICAgICBoZl9yZXBvX2lkPUhGX0NPTUZZX0gzX1JFUE8sCiAgICAgICAgaGZfcGF0aD0idGV4dF9lbmNvZGVycy9xd2VuM3ZsXzMyYl9taW5pbWF4X2gzX2JmMTYuc2FmZXRlbnNvcnMiLAogICAgICAgIHN1YmRpcj0idGV4dF9lbmNvZGVycyIsCiAgICAgICAgZmlsZW5hbWU9InF3ZW4zdmxfMzJiX21pbmltYXhfaDNfYmYxNi5zYWZldGVuc29ycyIsCiAgICAgICAgYXBwcm94X2dpYj00Ny45NywKICAgICAgICBsb2FkZXJfa2luZD0ic2FmZXRlbnNvcnMiLAogICAgICAgIGRlc2NyaXB0aW9uPSJRd2VuMy1WTC0zMkIgKEwwLTQ5KSBNaW5pTWF4LUgzIHRleHQvdmlzaW9uIGVuY29kZXIgKE5hdGl2ZSBCRjE2KSIsCiAgICApLAogICAgInZhZSI6IE1vZGVsU3BlYygKICAgICAgICBrZXk9InZhZSIsCiAgICAgICAgaGZfcmVwb19pZD1IRl9DT01GWV9JTUFHRV9SRVBPLAogICAgICAgIGhmX3BhdGg9InZhZS9taW5pbWF4X2gzX2ltYWdlX3ZhZV9mcDE2LnNhZmV0ZW5zb3JzIiwKICAgICAgICBzdWJkaXI9InZhZSIsCiAgICAgICAgZmlsZW5hbWU9Im1pbmltYXhfaDNfaW1hZ2VfdmFlX2ZwMTYuc2FmZXRlbnNvcnMiLAogICAgICAgIGFwcHJveF9naWI9MS4wOSwKICAgICAgICBsb2FkZXJfa2luZD0ic2FmZXRlbnNvcnMiLAogICAgICAgIGRlc2NyaXB0aW9uPSJNaW5pTWF4LUgzLUltYWdlIDI0LWNoIDE2eCBJbWFnZSBWQUUgKEZQMTYpIiwKICAgICksCiAgICAjIC0tLSBHR1VGIGNvbW11bml0eSBxdWFudHMgKEthZ2dsZSBHUFUgVDQgeDIgZmFsbGJhY2s6IDE2IEdpQiBWUkFNIC8gfjI5IEdpQiBSQU0pIC0tLQogICAgImRpdF9nZ3VmX3EzIjogTW9kZWxTcGVjKAogICAgICAgIGtleT0iZGl0X2dndWZfcTMiLAogICAgICAgIGhmX3JlcG9faWQ9SEZfR0dVRl9ESVRfUkVQTywKICAgICAgICBoZl9wYXRoPSJNaW5pTWF4LUgzLUltYWdlLVBydW5lZC1RM19LX00uZ2d1ZiIsCiAgICAgICAgc3ViZGlyPSJkaWZmdXNpb25fbW9kZWxzIiwKICAgICAgICBmaWxlbmFtZT0iTWluaU1heC1IMy1JbWFnZS1QcnVuZWQtUTNfS19NLmdndWYiLAogICAgICAgIGFwcHJveF9naWI9OC4zMCwKICAgICAgICBsb2FkZXJfa2luZD0iZ2d1ZiIsCiAgICAgICAgZGVzY3JpcHRpb249Ik1pbmlNYXgtSDMtSW1hZ2UgUHJ1bmVkIERpVCBHR1VGIFEzX0tfTSAoRlAzMiBzZW5zaXRpdmUgbGF5ZXJzIHByZXNlcnZlZCkiLAogICAgKSwKICAgICJkaXRfZ2d1Zl9xNCI6IE1vZGVsU3BlYygKICAgICAgICBrZXk9ImRpdF9nZ3VmX3E0IiwKICAgICAgICBoZl9yZXBvX2lkPUhGX0dHVUZfRElUX1JFUE8sCiAgICAgICAgaGZfcGF0aD0iTWluaU1heC1IMy1JbWFnZS1QcnVuZWQtUTRfS19NLmdndWYiLAogICAgICAgIHN1YmRpcj0iZGlmZnVzaW9uX21vZGVscyIsCiAgICAgICAgZmlsZW5hbWU9Ik1pbmlNYXgtSDMtSW1hZ2UtUHJ1bmVkLVE0X0tfTS5nZ3VmIiwKICAgICAgICBhcHByb3hfZ2liPTEwLjc4LAogICAgICAgIGxvYWRlcl9raW5kPSJnZ3VmIiwKICAgICAgICBkZXNjcmlwdGlvbj0iTWluaU1heC1IMy1JbWFnZSBQcnVuZWQgRGlUIEdHVUYgUTRfS19NIChGUDMyIHNlbnNpdGl2ZSBsYXllcnMgcHJlc2VydmVkKSIsCiAgICApLAogICAgImNsaXBfZ2d1Zl9pcTMiOiBNb2RlbFNwZWMoCiAgICAgICAga2V5PSJjbGlwX2dndWZfaXEzIiwKICAgICAgICBoZl9yZXBvX2lkPUhGX0dHVUZfQ0xJUF9SRVBPLAogICAgICAgIGhmX3BhdGg9IlF3ZW4zLVZMLTMyQi1JbnN0cnVjdC1NaW5pTWF4LUgzLUwwLTQ5LVVELUlRM19YWFMuZ2d1ZiIsCiAgICAgICAgc3ViZGlyPSJ0ZXh0X2VuY29kZXJzIiwKICAgICAgICBmaWxlbmFtZT0iUXdlbjMtVkwtMzJCLUluc3RydWN0LU1pbmlNYXgtSDMtTDAtNDktVUQtSVEzX1hYUy5nZ3VmIiwKICAgICAgICBhcHByb3hfZ2liPTkuMTAsCiAgICAgICAgbG9hZGVyX2tpbmQ9ImdndWYiLAogICAgICAgIGRlc2NyaXB0aW9uPSJRd2VuMy1WTC0zMkIgKEwwLTQ5KSBNaW5pTWF4LUgzIHRleHQgZW5jb2RlciBHR1VGIFVELUlRM19YWFMiLAogICAgKSwKICAgICJjbGlwX2dndWZfaXE0IjogTW9kZWxTcGVjKAogICAgICAgIGtleT0iY2xpcF9nZ3VmX2lxNCIsCiAgICAgICAgaGZfcmVwb19pZD1IRl9HR1VGX0NMSVBfUkVQTywKICAgICAgICBoZl9wYXRoPSJRd2VuMy1WTC0zMkItSW5zdHJ1Y3QtTWluaU1heC1IMy1MMC00OS1JUTRfWFMuZ2d1ZiIsCiAgICAgICAgc3ViZGlyPSJ0ZXh0X2VuY29kZXJzIiwKICAgICAgICBmaWxlbmFtZT0iUXdlbjMtVkwtMzJCLUluc3RydWN0LU1pbmlNYXgtSDMtTDAtNDktSVE0X1hTLmdndWYiLAogICAgICAgIGFwcHJveF9naWI9MTIuNTIsCiAgICAgICAgbG9hZGVyX2tpbmQ9ImdndWYiLAogICAgICAgIGRlc2NyaXB0aW9uPSJRd2VuMy1WTC0zMkIgKEwwLTQ5KSBNaW5pTWF4LUgzIHRleHQgZW5jb2RlciBHR1VGIElRNF9YUyIsCiAgICApLAogICAgImNsaXBfZ2d1Zl9tbXByb2oiOiBNb2RlbFNwZWMoCiAgICAgICAga2V5PSJjbGlwX2dndWZfbW1wcm9qIiwKICAgICAgICBoZl9yZXBvX2lkPUhGX0dHVUZfQ0xJUF9SRVBPLAogICAgICAgIGhmX3BhdGg9IlF3ZW4zLVZMLTMyQi1JbnN0cnVjdC1NaW5pTWF4LUgzLUwwLTQ5LW1tcHJvai1CRjE2LmdndWYiLAogICAgICAgIHN1YmRpcj0idGV4dF9lbmNvZGVycyIsCiAgICAgICAgZmlsZW5hbWU9IlF3ZW4zLVZMLTMyQi1JbnN0cnVjdC1NaW5pTWF4LUgzLUwwLTQ5LW1tcHJvai1CRjE2LmdndWYiLAogICAgICAgIGFwcHJveF9naWI9MS4xMiwKICAgICAgICBsb2FkZXJfa2luZD0iZ2d1ZiIsCiAgICAgICAgZGVzY3JpcHRpb249IlF3ZW4zLVZMLTMyQiBNaW5pTWF4LUgzIG11bHRpbW9kYWwgdmlzaW9uIHByb2plY3RvciAoYXV0by1sb2FkZWQgYnkgTmlmMDAvQ29tZnlVSS1HR1VGKSIsCiAgICApLAp9CgpGSUxFTkFNRV9UT19TUEVDOiBkaWN0W3N0ciwgTW9kZWxTcGVjXSA9IHtzcGVjLmZpbGVuYW1lOiBzcGVjIGZvciBzcGVjIGluIE1PREVMUy52YWx1ZXMoKX0KClBSRVNFVFM6IGRpY3Rbc3RyLCB0dXBsZVtzdHIsIC4uLl1dID0gewogICAgIyBQcmltYXJ5IFRQVSB2NWUtOCBzZXQgKH40NS45MSBHaUIpOiBJTlQ4IENvbnZSb3QgRGlUIChkZXF1YW50aXplZCB0byBCRjE2IGFjcm9zcyA4IGNoaXBzKSArIElOVDggUXdlbjMtVkwtMzJCICsgVkFFCiAgICAidHB1X2ludDgiOiAoImRpdF9pbnQ4IiwgImNsaXBfaW50OCIsICJ2YWUiKSwKICAgICMgTmF0aXZlIEJGMTYgRGlUIGZvciBUUFUgdjVlLTggKH42My44NCBHaUIpIHBhaXJlZCB3aXRoIElOVDggUXdlbjMtVkwtMzJCIG9uIDMzMCBHaUIgaG9zdCBSQU0KICAgICJ0cHVfYmYxNiI6ICgiZGl0X2JmMTYiLCAiY2xpcF9pbnQ4IiwgInZhZSIpLAogICAgIyBUNCB4MiBHR1VGIGJhbGFuY2VkIGZhbGxiYWNrICh+MjIuMDkgR2lCKTogUTRfS19NIERpVCArIElRM19YWFMgUXdlbjMtVkwtMzJCICsgbW1wcm9qICsgVkFFCiAgICAidDRfZ2d1ZiI6ICgiZGl0X2dndWZfcTQiLCAiY2xpcF9nZ3VmX2lxMyIsICJjbGlwX2dndWZfbW1wcm9qIiwgInZhZSIpLAogICAgIyBUNCB4MiBHR1VGIGxpZ2h0ZXN0IFJBTSBmb290cHJpbnQgKH4xOS42MSBHaUIpOiBRM19LX00gRGlUICsgSVEzX1hYUyBRd2VuMy1WTC0zMkIgKyBtbXByb2ogKyBWQUUKICAgICJ0NF9nZ3VmX2xpdGUiOiAoImRpdF9nZ3VmX3EzIiwgImNsaXBfZ2d1Zl9pcTMiLCAiY2xpcF9nZ3VmX21tcHJvaiIsICJ2YWUiKSwKICAgICMgVDQgeDIgR0dVRiBoaWdoZXItcXVhbGl0eSB0ZXh0IGVuY29kZXIgKH4yNS41MSBHaUI7IHRpZ2h0IG9uIDI5IEdpQiBob3N0IFJBTSkKICAgICJ0NF9nZ3VmX2hxIjogKCJkaXRfZ2d1Zl9xNCIsICJjbGlwX2dndWZfaXE0IiwgImNsaXBfZ2d1Zl9tbXByb2oiLCAidmFlIiksCn0KCkdHVUZfUFJFU0VUUzogZnJvemVuc2V0W3N0cl0gPSBmcm96ZW5zZXQoeyJ0NF9nZ3VmIiwgInQ0X2dndWZfbGl0ZSIsICJ0NF9nZ3VmX2hxIn0pCgpXT1JLRkxPV19URU1QTEFURV9VUkxTOiBkaWN0W3N0ciwgc3RyXSA9IHsKICAgICJpbWFnZV9taW5pbWF4X2gzX2ltYWdlX3QyaS5qc29uIjogKAogICAgICAgICJodHRwczovL3Jhdy5naXRodWJ1c2VyY29udGVudC5jb20vQ29tZnktT3JnL3dvcmtmbG93X3RlbXBsYXRlcy9tYWluL3RlbXBsYXRlcy8iCiAgICAgICAgImltYWdlX21pbmltYXhfaDNfaW1hZ2VfdDJpLmpzb24iCiAgICApLAogICAgImltYWdlX21pbmltYXhfaDNfaW1hZ2VfZWRpdC5qc29uIjogKAogICAgICAgICJodHRwczovL3Jhdy5naXRodWJ1c2VyY29udGVudC5jb20vQ29tZnktT3JnL3dvcmtmbG93X3RlbXBsYXRlcy9tYWluL3RlbXBsYXRlcy8iCiAgICAgICAgImltYWdlX21pbmltYXhfaDNfaW1hZ2VfZWRpdC5qc29uIgogICAgKSwKfQoKQ09NRllVSV9SRVBPX1VSTCA9ICJodHRwczovL2dpdGh1Yi5jb20vQ29tZnktT3JnL0NvbWZ5VUkuZ2l0IgojIEZvcmsgb2YgY2l0eTk2L0NvbWZ5VUktR0dVRiB3aXRoIGV4cGxpY2l0IHRydW5jYXRlZCBRd2VuMy1WTC0zMkIgKEwwLTQ5KSArIG1tcHJvaiBzdXBwb3J0IGZvciBNaW5pTWF4LUgzLgpHR1VGX05PREVfUkVQT19VUkwgPSAiaHR0cHM6Ly9naXRodWIuY29tL05pZjAwL0NvbWZ5VUktR0dVRi5naXQiCkdHVUZfTk9ERV9ESVJOQU1FID0gIkNvbWZ5VUktR0dVRiIKCkNMT1VERkxBUkVEX0xJTlVYX1VSTCA9ICgKICAgICJodHRwczovL2dpdGh1Yi5jb20vY2xvdWRmbGFyZS9jbG91ZGZsYXJlZC9yZWxlYXNlcy9sYXRlc3QvZG93bmxvYWQvY2xvdWRmbGFyZWQtbGludXgtYW1kNjQiCikKUlVOVElNRV9DQUNIRV9TTFVHID0gIm1pbmltYXgtaDMtcnVudGltZS1jYWNoZSIKUlVOVElNRV9XSEVFTFNfQVJDSElWRSA9ICJtaW5pbWF4X3Q0X3doZWVscy50YXIiClJVTlRJTUVfQ09NRllfQVJDSElWRSA9ICJjb21meXVpX3JlcG8udGFyIgpSVU5USU1FX0dHVUZfQVJDSElWRSA9ICJjb21meXVpX2dndWZfcmVwby50YXIiClJVTlRJTUVfQ0xPVURGTEFSRURfRklMRU5BTUUgPSAiY2xvdWRmbGFyZWQiClJVTlRJTUVfTUVUQV9GSUxFTkFNRSA9ICJtaW5pbWF4X3Q0X3J1bnRpbWVfbWV0YS5qc29uIgoKR1BVX0VYVFJBX1BJUF9QQUNLQUdFUzogdHVwbGVbc3RyLCAuLi5dID0gKAogICAgImh1Z2dpbmdmYWNlLWh1YltoZl94ZXRdPj0wLjMyLjAiLAogICAgImNvbWZ5LWtpdGNoZW4iLAogICAgImdndWY+PTAuMTMuMCIsCiAgICAic2VudGVuY2VwaWVjZSIsCiAgICAicHJvdG9idWYiLAogICAgImFpb2h0dHAiLAogICAgInBpbGxvdyIsCiAgICAibnVtcHkiLAogICAgInBhY2thZ2luZyIsCiAgICAicmVxdWVzdHMiLAogICAgIndyYXB0IiwKKQoKREVGQVVMVF9DT01GWV9ESVIgPSBQYXRoKCIvdG1wL0NvbWZ5VUkiKQpERUZBVUxUX1ZFTlZfRElSID0gUGF0aCgiL3RtcC92ZW52IikKREVGQVVMVF9IRl9DQUNIRV9ESVIgPSBQYXRoKCIvdG1wL2hmX2NhY2hlIikKREVGQVVMVF9XT1JLSU5HX0RJUiA9IFBhdGgoIi9rYWdnbGUvd29ya2luZyIpCkRFRkFVTFRfSU5QVVRfRElSID0gUGF0aCgiL2thZ2dsZS9pbnB1dCIpCgpTVEFOREFSRF9NT0RFTF9TVUJESVJTOiB0dXBsZVtzdHIsIC4uLl0gPSAoCiAgICAiY2hlY2twb2ludHMiLAogICAgImNsaXAiLAogICAgImNvbnRyb2xuZXQiLAogICAgImRpZmZ1c2lvbl9tb2RlbHMiLAogICAgImVtYmVkZGluZ3MiLAogICAgImxvcmFzIiwKICAgICJtb2RlbF9wYXRjaGVzIiwKICAgICJ0ZXh0X2VuY29kZXJzIiwKICAgICJ1bmV0IiwKICAgICJ1cHNjYWxlX21vZGVscyIsCiAgICAidmFlIiwKICAgICJ2YWVfYXBwcm94IiwKKQoKIyBPZmZpY2lhbCBNaW5pTWF4LUgzLUltYWdlIHNhbXBsaW5nIGNvbnRyYWN0IChDRkctZGlzdGlsbGVkIDYtc3RlcCBQREQgY2hlY2twb2ludCkKQ0FOVkFTX01VTFRJUExFID0gMzIKTUlOSU1BWF9ERUZBVUxUX1NURVBTID0gNgpNSU5JTUFYX0RFRkFVTFRfQ0ZHID0gMS4wCk1JTklNQVhfREVGQVVMVF9TQU1QTEVSID0gImV1bGVyIgpNSU5JTUFYX0RFRkFVTFRfU0NIRURVTEVSID0gInNpbXBsZSIKCiMgTWFzayBpbnBhaW50aW5nICYgb3V0cGFpbnRpbmcgZGVmYXVsdHMKREVGQVVMVF9JTlBBSU5UX01BU0tfRVhQQU5EID0gMTYKREVGQVVMVF9JTlBBSU5UX01BU0tfQkxVUiA9IDEyCkRFRkFVTFRfT1VUUEFJTlRfRkVBVEhFUiA9IDgwCgojIFNhZmUgcmVzb2x1dGlvbiBib3VuZHM6CiMgLSBUNCAoMTYgR2lCIFZSQU0pIGlzIGNhcHBlZCBhcm91bmQgNCBNUCAoMjA0OHgyMDQ4KSB0byBwcmV2ZW50IE9PTS4KIyAtIFRQVSB2NWUtOCAoMTI4IEdpQiBIQk0pIGRlZmF1bHRzIHRvIDJLLzRLICgxNiBNUCkgYW5kIGFsbG93cyB1cCB0byB+NDggTVAgZm9yIG11bHRpLXBlcnNvbiBncm91cCBmaWRlbGl0eS4KVDRfREVGQVVMVF9TSVpFID0gMTAyNApUNF9RVUFMSVRZX1NJWkUgPSAxNTM2ClQ0X01BWF9ESU1FTlNJT04gPSAyMDQ4ClRQVV9ERUZBVUxUX1NJWkUgPSAyMDQ4ClRQVV9RVUFMSVRZX1NJWkUgPSA0MDk2ClRQVV9NQVhfRElNRU5TSU9OID0gODE5MgpUUFVfTUFYX1BJWEVMUyA9IDQ5XzAwMF8wMDAgICMgdXAgdG8gfjQ4IE1QIChlLmcuIDY5MTJ4NjkxMj00Ny44TVAsIDgxOTJ4NTg1Nj00OC4wTVAsIDQwOTZ4NDA5Nj0xNi44TVApCgojIFNhZmUgZmxhZ3MgZm9yIEthZ2dsZSBUNCAoMTYgR2lCIFZSQU0sIH4zMSBHaUIgc3lzdGVtIFJBTSwgVHVyaW5nIHNtXzc1KS4KIyBSZWxpZXMgb24gQ29tZnlVSSBEeW5hbWljVlJBTSAoLS1sb3d2cmFtICsgY29tZnktYWltZG8pIHRvIHN0cmVhbSBHR1VGIGxheWVycyB3aXRob3V0CiMgYmxvYXRpbmcgaG9zdCBSQU0gKH4zLjggR2lCIFJTUyBvYnNlcnZlZCBsaXZlKSwgd2hpbGUga2VlcGluZyBDb21meVVJJ3MgZGVmYXVsdCBub2RlIGNhY2hlCiMgc28gMjIgR2lCIG9mIEdHVUYgd2VpZ2h0cyBhbmQgdG9rZW5fZW1iZCBkbyBub3QgcmVsb2FkIGZyb20gc2NyYXRjaCBvbiBldmVyeSBwcm9tcHQuClQ0X1NBRkVfQ09NRllfRkxBR1M6IHR1cGxlW3N0ciwgLi4uXSA9ICgKICAgICItLWxvd3ZyYW0iLAogICAgIi0tZGlzYWJsZS1waW5uZWQtbWVtb3J5IiwKICAgICItLWRpc2FibGUtYXN5bmMtb2ZmbG9hZCIsCiAgICAiLS1wcmV2aWV3LW1ldGhvZCIsCiAgICAibGF0ZW50MnJnYiIsCiAgICAiLS1kaXNhYmxlLWF1dG8tbGF1bmNoIiwKKQoKIyBTYWZlIGZsYWdzIGZvciBLYWdnbGUgVFBVIHY1ZS04ICgxMjggR2lCIEhCTSBhY3Jvc3MgOCBjaGlwcyArIH4zMzAgR2lCIGhvc3QgUkFNKS4KVFBVX1NBRkVfQ09NRllfRkxBR1M6IHR1cGxlW3N0ciwgLi4uXSA9ICgKICAgICItLWJmMTYtdW5ldCIsCiAgICAiLS1kaXNhYmxlLXBpbm5lZC1tZW1vcnkiLAogICAgIi0tZGlzYWJsZS1hc3luYy1vZmZsb2FkIiwKICAgICItLXByZXZpZXctbWV0aG9kIiwKICAgICJsYXRlbnQycmdiIiwKICAgICItLWRpc2FibGUtYXV0by1sYXVuY2giLAopCgpUUFVfQ09OVEVYVF9CVUNLRVRTOiB0dXBsZVtpbnQsIC4uLl0gPSAoMjU2LCA1MTIsIDEwMjQsIDIwNDgpCg==",
  "workflows.py": "...",
  "bootstrap.py": "...",
  "smoke_test.py": "...",
  "prep_dataset.py": "...",
  "tpu_runtime.py": "...",
  "gateway.py": "..."
}

for name, b64 in _MODULES_B64.items():
    (PKG_DIR / name).write_bytes(base64.b64decode(b64))

for mod_key in [k for k in list(sys.modules) if k == 'minimax_kaggle' or k.startswith('minimax_kaggle.')]:
    sys.modules.pop(mod_key, None)

if str(PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(PKG_ROOT))

print(f'Unpacked {len(_MODULES_B64)} minimax_kaggle modules into {PKG_DIR}')


In [ ]:
from minimax_kaggle.constants import DEFAULT_COMFY_DIR, DEFAULT_INPUT_DIR, DEFAULT_VENV_DIR
from minimax_kaggle.gateway import prepare_and_launch_gateway_subprocess

# --- Konfiguracja serwera Kaggle T4 x2 (GGUF) ---
MODEL_PRESET = 't4_gguf'   # 't4_gguf' (~22.1 GiB) lub 't4_gguf_lite' (~19.6 GiB)
NUM_WORKERS = 2            # 2 workery rownolegle GPU 0 + GPU 1 (zweryfikowane live: 113.5s za 2 obrazy, peak RAM 5.92 GiB)
PORT = 8000
USE_TUNNEL = True
IDLE_TIMEOUT_S = 600       # auto-stop po 10 min bez zadan (0 = wylaczone)
PROMPT_TIMEOUT_S = 2400    # limit jednego zlecenia w sek.
FORCE_HF = False           # True ignoruje /kaggle/input
BEARER_TOKEN = ''          # Opcjonalny token (lub sekret MINIMAX_GATEWAY_TOKEN / QWEN_GATEWAY_TOKEN)

prepare_and_launch_gateway_subprocess(
    comfy_dir=DEFAULT_COMFY_DIR,
    venv_dir=DEFAULT_VENV_DIR,
    input_dir=DEFAULT_INPUT_DIR,
    preset=MODEL_PRESET,
    port=PORT,
    num_workers=NUM_WORKERS,
    force_hf=FORCE_HF,
    bearer_token=BEARER_TOKEN,
    use_tunnel=USE_TUNNEL,
    accelerator='gpu',
    idle_timeout_s=IDLE_TIMEOUT_S,
    prompt_timeout_s=PROMPT_TIMEOUT_S,
)
